# Prepare task

In [84]:
import pandas as pd
import os

def get_fire_info(event_id, file_path = 'datasets/FEDS25MTBS/fireslist2012-2023.csv'):
    assert os.path.exists(file_path), f"Error: File not found at {file_path}"
    
    df = pd.read_csv(file_path, dtype={'Event_ID': str})
    df.columns = df.columns.str.strip()
    
    df['tst'] = pd.to_datetime(df['tst'], errors='coerce')
    df['ted'] = pd.to_datetime(df['ted'], errors='coerce')
    
    fire_row = df[df['Event_ID'] == event_id]
    
    assert not fire_row.empty, f"Error: Event_ID {event_id} not found in the dataset"
    
    row = fire_row.iloc[0]
    
    return {
        'event_id': row['Event_ID'],
        'year': row['Year'],
        'name': row['Incid_Name'],
        'bounds': [row['lon0'], row['lat0'], row['lon1'], row['lat1']],
        't_start': row['tst'].to_pydatetime(),
        't_end': row['ted'].to_pydatetime(),
    }

In [85]:
fire_info = get_fire_info('CA3982012144020181108')
print(fire_info)

{'event_id': 'CA3982012144020181108', 'year': np.int64(2018), 'name': 'CAMP', 'bounds': [np.float64(-121.77781464622892), np.float64(39.598580232291965), np.float64(-121.3525918499339), np.float64(39.897810492060614)], 't_start': datetime.datetime(2018, 11, 8, 0, 0), 't_end': datetime.datetime(2018, 11, 20, 12, 0)}


In [86]:
from shapely.geometry import box
import math
import geopandas as gpd

def get_task_info(crs: str = 'EPSG:3310', buffering: int = 180, resolution: int = 30, fire_info: dict = None):
    
    event_id = fire_info.get('event_id')
    year = fire_info.get('year')
    name = fire_info.get('name')
    bounds = fire_info.get('bounds')
    t_start = fire_info.get('t_start')
    t_end = fire_info.get('t_end')
    
    minx, miny, maxx, maxy = bounds
    bbox_poly = box(minx, miny, maxx, maxy)
    bounds_gs = gpd.GeoSeries([bbox_poly], crs="EPSG:4326")
    bounds_proj = bounds_gs.to_crs(crs)
    
    bounds_proj = bounds_proj.buffer(buffering)
    
    
    t_minx, t_miny, t_maxx, t_maxy = bounds_proj.total_bounds

    target_bounds = [
        math.floor(t_minx / resolution) * resolution,
        math.floor(t_miny / resolution) * resolution,
        math.ceil(t_maxx / resolution) * resolution,
        math.ceil(t_maxy / resolution) * resolution
    ]
    
    return {
        'event_id': event_id,
        'year': year,
        'name': name,
        'resolution': resolution,
        'crs': crs,
        'target_bounds': target_bounds,
        't_start': t_start,
        't_end': t_end,
    }

In [87]:
task = get_task_info(fire_info=fire_info)
print(task)

{'event_id': 'CA3982012144020181108', 'year': np.int64(2018), 'name': 'CAMP', 'resolution': 30, 'crs': 'EPSG:3310', 'target_bounds': [-152760, 176490, -115410, 210720], 't_start': datetime.datetime(2018, 11, 8, 0, 0), 't_end': datetime.datetime(2018, 11, 20, 12, 0)}


# FIRE DATA from FEDS25MTBS

In [88]:
import os
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPolygon, Polygon

def extract_fire_data(fire_info, base_dir='datasets/FEDS25MTBS'):

    event_id = fire_info.get('event_id')
    year = fire_info.get('year')
    file_path = f"{base_dir}/{year}/{event_id}.gpkg"

    assert os.path.exists(file_path), f"Error: Geopackage file not found at {file_path}"

    gdf = gpd.read_file(file_path, layer='perimeter')

    data_list = []
    timestamps = []

    for _, row in gdf.iterrows():
        timestamp = pd.to_datetime(row['t'])
        
        geom = row.geometry

        if geom is None:
            continue

        if isinstance(geom, Polygon):
            geom = MultiPolygon([geom])
        
        if isinstance(geom, MultiPolygon):
            data_list.append(geom)
            timestamps.append(timestamp)

    return {
        'data': data_list,
        'timestamps': timestamps,
        'metadata': {
            'name': 'FEDS25MTBS'
        }
    }

In [89]:
import os
import numpy as np
import geopandas as gpd
from rasterio import features
from rasterio.transform import from_origin
from shapely.geometry import box

def process_and_save_fire_data(task):
    # 1. Extract Data
    raw_data = extract_fire_data(task)
    
    # 2. Setup Output Directory
    event_id = task['event_id']
    output_dir = os.path.join('./output', event_id)
    os.makedirs(output_dir, exist_ok=True)
    
    # 3. Calculate Grid Dimensions
    t_minx, t_miny, t_maxx, t_maxy = task['target_bounds']
    
    res = task['resolution']
    width = int(np.ceil((t_maxx - t_minx) / res))
    height = int(np.ceil((t_maxy - t_miny) / res))
    
    transform = from_origin(t_minx, t_maxy, res, res)
    
    print(f"Target Grid: {width}x{height} pixels @ {res}m resolution")
    
    processed_rasters = []
    
    # 5. Process each timestep
    gdf = gpd.GeoDataFrame({
        'geometry': raw_data['data'],
        'timestamp': raw_data['timestamps']
    }, crs="EPSG:4326") # Assuming raw data comes in Lat/Lon
    
    # Reproject to target CRS
    print(f"Reprojecting to {task['crs']}...")
    gdf = gdf.to_crs(task['crs'])
    
    # Rasterize
    print("Rasterizing...")
    for idx, row in gdf.iterrows():
        # Rasterize creates a numpy array where geometry exists
        # We burn a value of '1' where the polygon is, '0' otherwise
        shapes = [(row.geometry, 1)]
        
        raster = features.rasterize(
            shapes=shapes,
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype=np.uint8,
            all_touched=True # If a pixel touches the polygon, it counts
        )
        processed_rasters.append(raster)
        
    # Stack into a numpy array (Time, Height, Width) or keep as list
    # The prompt asks for a "numpy array", typically for time series this is stacked
    data_array = np.array(processed_rasters)
    
    # 6. Construct Final Output Dictionary
    output_payload = {
        'data': data_array, # The rasterized 3D numpy array
        'timestamps': raw_data['timestamps'],
        'metadata': raw_data['metadata']
    }
    
    # 7. Save to .npy
    output_path = os.path.join(output_dir, 'fire_data.npy')
    np.save(output_path, output_payload)
    
    print(f"Success! Data saved to: {output_path}")
    print(f"Output Shape: {data_array.shape}")


In [90]:
process_and_save_fire_data(task)

Target Grid: 1245x1141 pixels @ 30m resolution
Reprojecting to EPSG:3310...
Rasterizing...
Success! Data saved to: ./output/CA3982012144020181108/fire_data.npy
Output Shape: (25, 1141, 1245)


# ELEVATION

In [91]:
import ee
import os
import io
import requests
import numpy as np
import rasterio

def process_gee_task(task, imagecollection, data_name):
    # 1. Initialize Earth Engine
    try:
        ee.Initialize()
    except Exception as e:
        print("Earth Engine not initialized. Attempting to authenticate...")
        ee.Authenticate()
        ee.Initialize()

    event_id = task['event_id']
    
    print(f"Processing Elevation for Event: {task['name']} ({event_id})")

    # 2. Define Geometry
    bounds = task['target_bounds']
    roi = ee.Geometry.Rectangle([bounds[0], bounds[1], bounds[2], bounds[3]], proj=task['crs'], geodesic=False)
    
    # 3. Select Data
    # 1m data is an ImageCollection, so we mosaic it to get a single seamless image.
    elev_collection = ee.ImageCollection(imagecollection)
    elev_image = elev_collection.filterBounds(roi).mosaic()
    
    # 4. Prepare Download URL
    # We specify the CRS, Scale, and Region here. GEE handles the 
    # reprojection (Resampling) and Rasterization server-side before download.
    download_params = {
        'name': f'{data_name}_{event_id}',
        'crs': task['crs'],
        'scale': task['resolution'],
        'region': roi, # The download region is the original geometry
        'format': 'GEO_TIFF'
    }

    try:
        url = elev_image.getDownloadURL(download_params)
        print(f"Download URL generated. Fetching data...")
        print(url)
    except Exception as e:
        print(f"Error generating download URL (Area might be too large): {e}")
        return

    # 5. Download and Read into Memory
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"HTTP Error {response.status_code}: {response.text}")
        response.raise_for_status()

    # Check content type to handle both ZIP (default) and raw TIFF (small areas)
    content_type = response.headers.get('Content-Type', '')
    print(f"Received content type: {content_type}")

    try:
        # It is a raw GeoTIFF
        with rasterio.open(io.BytesIO(response.content)) as src:
            data = src.read(1)
                    
    except (rasterio.errors.RasterioIOError):
        print("\n--- ERROR DOWNLOADING DATA ---")
        print("The server returned a file that could not be processed as TIFF.")
        print(f"Content-Type received: {response.headers.get('Content-Type')}")
        print("Response content (First 500 bytes):")
        print(response.content[:500])
        print("------------------------------")
        raise

    # 6. Format Output
    # The prompt requests a specific dictionary structure.
    output_payload = {
        'data': data, # The numpy array
        'timestamps': [fire_info['t_start']], # Using start time as the anchor timestamp
        'metadata': {
            'name': data_name,
            'crs': task['crs'],
            'resolution': task['resolution'],
            'source': imagecollection,
        }
    }

    # 7. Save to File
    output_dir = os.path.join('./output', event_id)
    os.makedirs(output_dir, exist_ok=True)
    
    save_path = os.path.join(output_dir, f'{data_name}.npy')
    
    # Save as a pickled numpy dictionary
    np.save(save_path, output_payload)
    
    print(f"Success. Data shape: {data.shape}")
    print(f"Saved to: {save_path}")


In [92]:
def process_elevation_task(task):
    return process_gee_task(task, 'USGS/3DEP/1m', data_name='elevation')


In [93]:
process_elevation_task(task)

Processing Elevation for Event: CAMP (CA3982012144020181108)
Download URL generated. Fetching data...
https://earthengine.googleapis.com/v1/projects/32254342601/thumbnails/b23947d1eea91639aa69ce560d374166-765b0bdcc9d9298a10c3715540976a9f:getPixels
Received content type: image/tiff
Success. Data shape: (1141, 1245)
Saved to: ./output/CA3982012144020181108/elevation.npy


# LANDFIRE

In [94]:
def process_cbd_task(task):
    process_gee_task(task, 'projects/sat-io/open-datasets/landfire/FUEL/CBD', data_name='cbd')


In [95]:
process_cbd_task(task)

Processing Elevation for Event: CAMP (CA3982012144020181108)
Download URL generated. Fetching data...
https://earthengine.googleapis.com/v1/projects/32254342601/thumbnails/e2979d42a62ed8c2fd9357cbd1ba1f38-43daecfda636d62410694f524c3b1160:getPixels
Download URL generated. Fetching data...
https://earthengine.googleapis.com/v1/projects/32254342601/thumbnails/e2979d42a62ed8c2fd9357cbd1ba1f38-43daecfda636d62410694f524c3b1160:getPixels
Received content type: image/tiff
Success. Data shape: (1141, 1245)
Saved to: ./output/CA3982012144020181108/cbd.npy
Received content type: image/tiff
Success. Data shape: (1141, 1245)
Saved to: ./output/CA3982012144020181108/cbd.npy


In [96]:
def process_cc_task(task):
    process_gee_task(task, 'projects/sat-io/open-datasets/landfire/FUEL/CC', data_name='cc')


In [97]:
process_cc_task(task)

Processing Elevation for Event: CAMP (CA3982012144020181108)
Download URL generated. Fetching data...
https://earthengine.googleapis.com/v1/projects/32254342601/thumbnails/a99b6e684559eaeddcb00136333faa61-99f9205a5fc271820b6f749924531234:getPixels
Download URL generated. Fetching data...
https://earthengine.googleapis.com/v1/projects/32254342601/thumbnails/a99b6e684559eaeddcb00136333faa61-99f9205a5fc271820b6f749924531234:getPixels
Received content type: image/tiff
Success. Data shape: (1141, 1245)
Saved to: ./output/CA3982012144020181108/cc.npy
Received content type: image/tiff
Success. Data shape: (1141, 1245)
Saved to: ./output/CA3982012144020181108/cc.npy


# HRRR

In [98]:
import os
import datetime
import numpy as np
import xarray as xr
import rioxarray
import pyproj
from herbie import Herbie
import pandas as pd

# ---------------------------------------------------------
# 1. Helper Functions (Coordinate Fix & Geospatial Processing)
# ---------------------------------------------------------

def fix_hrrr_coords(ds):
    """
    Calculates 1D x/y coordinates in meters from the 2D lat/lon arrays 
    and assigns them to the dataset so rioxarray can read the grid.
    """
    # 1. Ensure we have the CRS object loaded from metadata
    if ds.rio.crs is None:
        try:
            # HRRR usually has a variable like 'gribfile_projection' or 'unknown'
            proj_var = [v for v in ds.data_vars if 'projection' in v]
            if proj_var:
                ds = ds.rio.write_crs(ds[proj_var[0]].attrs)
        except Exception:
            pass

    if ds.rio.crs is None:
        # Fallback for HRRR (Lambert Conformal) if metadata fails
        # This is the standard HRRR projection parameter set
        ds = ds.rio.write_crs("+proj=lcc +lat_1=38.5 +lat_2=38.5 +lat_0=38.5 +lon_0=-97.5 +x_0=0 +y_0=0 +a=6371229 +b=6371229 +units=m +no_defs")

    # 2. Create a transformer from Lat/Lon (EPSG:4326) to the HRRR Projected CRS
    transformer = pyproj.Transformer.from_crs("EPSG:4326", ds.rio.crs, always_xy=True)

    # 3. Transform the 2D Lat/Lon arrays to Projected X/Y
    # HRRR grib imports usually result in 'latitude' and 'longitude' coordinates
    xx, yy = transformer.transform(ds.longitude.values, ds.latitude.values)

    # 4. Extract 1D vectors
    # Since HRRR is a rectilinear grid in its own projection, 
    # x is constant along columns, y is constant along rows.
    x_1d = xx[0, :]
    y_1d = yy[:, 0]

    # 5. Assign these new coordinates to the x and y dimensions
    # Rename dimensions if they are strictly 'y' and 'x' in the GRIB import
    ds = ds.assign_coords(x=x_1d, y=y_1d)
    
    return ds

def process_geospatial(ds, target_crs, bounds):
    """
    Standardizes coords, reprojects, and clips the dataset.
    """
    # Standardize Dimension Names if needed (grib often loads as y, x)
    if 'latitude' in ds.dims and 'longitude' in ds.dims:
        ds = ds.rename({'longitude': 'x', 'latitude': 'y'})
    
    # Fix Coordinates
    ds = fix_hrrr_coords(ds)
    
    # Reproject
    ds_reproj = ds.rio.reproject(target_crs)

    # Crop
    ds_cropped = ds_reproj.rio.clip_box(
        minx=bounds[0],
        miny=bounds[1],
        maxx=bounds[2],
        maxy=bounds[3]
    )
    
    return ds_cropped

# ---------------------------------------------------------
# 2. Main Processing Logic
# ---------------------------------------------------------

def process_hrrr_task(task):
    print(f"Processing Task: {task['name']} ({task['event_id']})")
    
    # Setup Output Directory
    output_dir = f"./output/{task['event_id']}/"
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize storage lists for aggregation
    data_buffer = {
        'r2': [],  # Humidity
        'u10': [], # Wind U
        'v10': []  # Wind V
    }
    timestamps = []

    # Generate list of hours to process
    # Creates a range from start to end (inclusive)
    current_time = task['t_start']
    end_time = task['t_end']
    
    while current_time <= end_time:
        print(f"Downloading HRRR for: {current_time}")
        
        try:
            # Initialize Herbie
            H = Herbie(
                current_time,
                model='hrrr',
                product='sfc',
                fxx=0,
                verbose=False # Set to True if you want Herbie logs
            )
            
            # --- 1. Process Humidity ---
            ds_rh = H.xarray(":RH:2 m", remove_grib=False)
            ds_rh_proc = process_geospatial(ds_rh, task['crs'], task['target_bounds'])
            
            # --- 2. Process Wind ---
            ds_wind = H.xarray(":(?:UGRD|VGRD):10 m", remove_grib=False)
            ds_wind_proc = process_geospatial(ds_wind, task['crs'], task['target_bounds'])
            
            # Append data to buffers
            # We use .values to get the numpy array of the 2D slice
            # variable names usually match the cfgrib shortNames
            data_buffer['r2'].append(ds_rh_proc['r2'].values)
            data_buffer['u10'].append(ds_wind_proc['u10'].values)
            data_buffer['v10'].append(ds_wind_proc['v10'].values)
            
            timestamps.append(current_time)

        except Exception as e:
            print(f"Failed to process {current_time}: {e}")
        
        # Increment time by 1 hour
        current_time += datetime.timedelta(hours=1)

    # ---------------------------------------------------------
    # 3. Aggregation and Saving
    # ---------------------------------------------------------
    
    if not timestamps:
        print("No data processed.")
        return

    # Convert timestamp list to numpy array of datetime64
    ts_array = np.array(timestamps, dtype='datetime64[ns]')

    # Save function to keep things clean
    def save_variable(var_name, save_name):
        # Stack list of 2D arrays into a 3D array (Time, Y, X)
        stacked_data = np.stack(data_buffer[var_name], axis=0)
        
        output_dict = {
            'data': stacked_data,
            'timestamps': ts_array,
            'metadata': {
                'name': 'FEDS25MTBS'
            }
        }
        
        filename = os.path.join(output_dir, save_name)
        np.save(filename, output_dict)
        print(f"Saved {save_name} with shape {stacked_data.shape}")

    # Execute Saves
    save_variable('r2', 'humidity.npy')
    save_variable('u10', 'wind_u.npy')
    save_variable('v10', 'wind_v.npy')

    print(f"Task completed. Output saved to {output_dir}")


In [99]:

# Define the task
task = {
    'event_id': 'CA3982012144020181108',
    'year': np.int64(2018),
    'name': 'CAMP',
    'resolution': 30,
    'crs': 'EPSG:3310',
    # [minx, miny, maxx, maxy]
    'target_bounds': [-152760, 176490, -115410, 210720], 
    't_start': datetime.datetime(2018, 11, 8, 0, 0),
    't_end': datetime.datetime(2018, 11, 8, 4, 0)
}

# Run
process_hrrr_task(task)

Processing Task: CAMP (CA3982012144020181108)
Saved humidity.npy with shape (5, 12, 13)
Saved wind_u.npy with shape (5, 12, 13)
Saved wind_v.npy with shape (5, 12, 13)
Task completed. Output saved to ./output/CA3982012144020181108/
Saved humidity.npy with shape (5, 12, 13)
Saved wind_u.npy with shape (5, 12, 13)
Saved wind_v.npy with shape (5, 12, 13)
Task completed. Output saved to ./output/CA3982012144020181108/


# Urban Data

In [100]:
# Height, TBD from GEE